# VAE on MNIST

Variational Autoencoder trained from scratch on MNIST.

**What this notebook demonstrates:**
1. Training a VAE with ELBO loss (BCE reconstruction + KL regularisation)
2. Reconstruction quality vs latent dimension
3. Continuous latent space interpolation ("latent space walk")
4. Reconstruction × KL tradeoff by varying $\beta$ in the ELBO

**Key mathematical idea:**
$$\mathcal{L}_{\text{VAE}} = -\mathbb{E}_{z \sim q_\phi(z|x)}[\log p_\theta(x|z)] + \text{KL}(q_\phi(z|x) \parallel p(z))$$

where $q_\phi(z|x) = \mathcal{N}(\mu_\phi(x), \sigma_\phi^2(x))$ is the encoder and $p_\theta(x|z)$ is the decoder.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Project imports
p = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if p not in sys.path:
    sys.path.insert(0, p)

from core.gen import VAE

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128
EPOCHS = 50
LR = 1e-3
LATENT_DIM = 20  # bottleneck size (try varying: 2, 10, 20, 50)

torch.manual_seed(42)
np.random.seed(42)
print(f"Device: {DEVICE}, Latent dim: {LATENT_DIM}")

### Data — MNIST

Normalised to $[0, 1]$ for BCE loss (each pixel treated as a Bernoulli probability).


In [ ]:
DATA_ROOT = (
    Path.cwd().parent / "assets" if Path.cwd().name == "apps" else Path.cwd() / "assets"
)

transform = transforms.Compose([transforms.ToTensor()])  # already scales to [0, 1]

train_set = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=transform)
test_set = datasets.MNIST(DATA_ROOT, train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_set)}  |  Test: {len(test_set)}")

In [ ]:
# Sample a few training images
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(imgs[i].squeeze(), cmap="gray")
    ax.set_title(f"label={labels[i].item()}", fontsize=9)
    ax.axis("off")
fig.suptitle("MNIST samples");

### Model — VAE

The VAE consists of:
- **Encoder**: Conv → Conv → Linear → (μ, log σ²) — parameterises $q_\phi(z|x)$
- **Decoder**: Linear → ConvTranspose → ConvTranspose → Sigmoid — computes $p_\theta(x|z)$

The latent code $z$ is sampled via the **reparameterisation trick**:
$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$


In [ ]:
model = VAE(
    in_channels=1,
    latent_dim=LATENT_DIM,
    hidden_dim=256,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"VAE: {total_params / 1e3:.1f}K parameters")
print(model)

In [ ]:
# ── Optimizer ──────────────────────────────────────────────────────────
optimizer = optim.Adam(model.parameters(), lr=LR)

### Training loop

Each step:
1. Forward: `model(x)` → `(recon, mu, logvar)`
2. Loss: `model.loss_function(recon, x, mu, logvar)` → `{loss, recon_loss, kl_loss}`
3. Backward: `loss.backward()` → `optimizer.step()`


In [ ]:
def train_epoch(model, loader, optimizer, beta=1.0):
    """Train for one epoch.  beta > 1 strengthens KL, beta < 1 weakens it."""
    model.train()
    total_loss = 0.0
    recon_loss_sum = 0.0
    kl_loss_sum = 0.0
    n = 0
    for x, _ in loader:
        x = x.to(DEVICE)
        optimizer.zero_grad()
        recon, mu, logvar = model(x)

        loss_dict = model.loss_function(recon, x, mu, logvar)
        loss = loss_dict["loss"]
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        recon_loss_sum += loss_dict["recon_loss"].item() * bs
        kl_loss_sum += loss_dict["kl_loss"].item() * bs
        n += bs
    return {
        "loss": total_loss / n,
        "recon_loss": recon_loss_sum / n,
        "kl_loss": kl_loss_sum / n,
    }


@torch.no_grad()
def eval_loss(model, loader, beta=1.0):
    """Compute loss on held-out set."""
    model.eval()
    total_loss = 0.0
    recon_loss_sum = 0.0
    kl_loss_sum = 0.0
    n = 0
    for x, _ in loader:
        x = x.to(DEVICE)
        recon, mu, logvar = model(x)
        loss_dict = model.loss_function(recon, x, mu, logvar)
        bs = x.size(0)
        total_loss += loss_dict["loss"].item() * bs
        recon_loss_sum += loss_dict["recon_loss"].item() * bs
        kl_loss_sum += loss_dict["kl_loss"].item() * bs
        n += bs
    return {
        "loss": total_loss / n,
        "recon_loss": recon_loss_sum / n,
        "kl_loss": kl_loss_sum / n,
    }

In [ ]:
train_metrics = {k: [] for k in ["loss", "recon_loss", "kl_loss"]}
test_metrics = {k: [] for k in ["loss", "recon_loss", "kl_loss"]}

for epoch in range(EPOCHS):
    train_m = train_epoch(model, train_loader, optimizer)
    test_m = eval_loss(model, test_loader)

    for k in train_metrics:
        train_metrics[k].append(train_m[k])
        test_metrics[k].append(test_m[k])

    print(
        f"Epoch {epoch + 1:2d} | "
        f"train loss {train_m['loss']:.3f} (recon {train_m['recon_loss']:.3f} + KL {train_m['kl_loss']:.3f}) | "
        f"test loss {test_m['loss']:.3f}"
    )

### 1. Training Curves

Monitor convergence: total ELBO, reconstruction loss, and KL divergence.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

titles = ["Total ELBO Loss", "Reconstruction Loss (BCE)", "KL Divergence"]
keys = ["loss", "recon_loss", "kl_loss"]

for ax, title, key in zip(axes, titles, keys):
    ax.plot(train_metrics[key], label="train")
    ax.plot(test_metrics[key], label="test")
    ax.set_xlabel("Epoch")
    ax.set_title(title)
    ax.legend()

fig.suptitle(f"VAE training — latent_dim={LATENT_DIM}", fontsize=12)
plt.tight_layout()

### 2. Reconstructions

Original images (top row) vs VAE reconstructions (bottom row).

The reconstruction degrades when you shrink the latent bottleneck — this is the **information bottleneck** in action.


In [ ]:
@torch.no_grad()
def show_reconstructions(model, loader, n=10):
    """Show original (top) and reconstructed (bottom) images."""
    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(DEVICE)
    recon, _, _ = model(x)

    fig, axes = plt.subplots(2, n, figsize=(n * 1.5, 3))
    for i in range(n):
        # Original
        axes[0, i].imshow(x[i].cpu().squeeze(), cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_ylabel("Original", fontsize=10)

        # Reconstruction
        axes[1, i].imshow(recon[i].cpu().squeeze(), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_ylabel("Recon", fontsize=10)

    fig.suptitle("Original (top) vs Reconstruction (bottom)", fontsize=12)
    plt.tight_layout()


show_reconstructions(model, test_loader)

### 3. Latent Space Walk

Interpolate between two random test images in latent space and decode
every intermediate point.  A well-trained VAE produces **smooth,
semantically meaningful** transitions — digits morph gradually
through intermediate shapes rather than jumping or producing noise.

This is the key test of whether the latent space is **continuous**
and **complete** (no "holes").


In [ ]:
@torch.no_grad()
def latent_walk(model, loader, n_steps=12):
    """Interpolate between two random test images in latent space."""
    model.eval()
    x, _ = next(iter(loader))
    x = x[:2].to(DEVICE)

    # Encode both images to get their latent means
    mu_1, _ = model.encoder(x[0:1])
    mu_2, _ = model.encoder(x[1:2])

    # Linear interpolation in latent space: z = (1 - α)·z₁ + α·z₂
    alphas = torch.linspace(0, 1, n_steps, device=DEVICE)
    interp_recons = []
    for a in alphas:
        z = (1 - a) * mu_1 + a * mu_2
        recon = model.decoder(z)
        interp_recons.append(recon)

    # ── Plot ─────────────────────────────────────────────────────
    # Layout: original_1 | α=0 → α=1 interpolated sequence | original_2
    fig, axes = plt.subplots(1, n_steps + 2, figsize=((n_steps + 2) * 1.5, 2.5))

    # Original image 1
    axes[0].imshow(x[0].cpu().squeeze(), cmap="gray")
    axes[0].set_title("orig", fontsize=9)
    axes[0].axis("off")

    # Interpolation sequence
    for i, a in enumerate(alphas):
        axes[i + 1].imshow(interp_recons[i].cpu().squeeze(), cmap="gray")
        axes[i + 1].set_title(f"α={a:.2f}", fontsize=8)
        axes[i + 1].axis("off")

    # Original image 2
    axes[-1].imshow(x[1].cpu().squeeze(), cmap="gray")
    axes[-1].set_title("orig", fontsize=9)
    axes[-1].axis("off")

    fig.suptitle(
        "Latent space walk: smooth interpolation between two MNIST digits", fontsize=12
    )
    plt.tight_layout()


latent_walk(model, test_loader)

### 4. Random Generation (Prior Sampling)

Sample $z \sim \mathcal{N}(0, I)$ from the prior and decode.
The decoder learns to map **any** point near the origin to a plausible image —
this is what makes VAE a *generative* model.


In [ ]:
@torch.no_grad()
def generate_samples(model, n=16):
    """Generate images by sampling from the prior p(z) = N(0, I)."""
    model.eval()

    # 1. Sample z ~ N(0, I) — standard normal prior
    z = torch.randn(n, model.latent_dim, device=next(model.parameters()).device)

    # 2. Decode: p(x|z)
    samples = model.decoder(z)  # (n, 1, 28, 28)

    # 3. Plot as a 4×4 grid
    fig, axes = plt.subplots(4, 4, figsize=(5, 5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i].cpu().squeeze(), cmap="gray")
        ax.axis("off")
    fig.suptitle("Random generation: z ~ N(0, I) → decoder", fontsize=12)
    plt.tight_layout()


generate_samples(model)

### 5. Reconstruction $\times$ KL Tradeoff ($\beta$-VAE)

The ELBO loss has two competing terms:
$$\mathcal{L} = \underbrace{\text{BCE}(x, \hat{x})}_{\text{reconstruction}} + \beta \cdot \underbrace{\text{KL}(q(z|x) \parallel \mathcal{N}(0, I))}_{\text{regularisation}}$$

- **$\beta = 0$**: pure autoencoder — great reconstruction, but latent space has holes (bad for generation)
- **$\beta = 1$**: standard VAE — balances recon and KL
- **$\beta > 1$** ($\beta$-VAE) — stronger regularisation, better disentanglement but blurrier reconstructions

The cell below trains several copies of the VAE with different $\beta$ values
and plots the Pareto frontier (reconstruction error vs KL divergence).


In [ ]:
# Train VAE with different beta values and compare.
# Plot recon_loss vs kl_loss to see the Pareto frontier.

BETA_EPOCHS = 10  # fewer epochs per run for speed

betas = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]

beta_results = {"beta": [], "recon_loss": [], "kl_loss": []}

for beta in betas:
    # 1. Fresh VAE with same architecture
    model_beta = VAE(in_channels=1, latent_dim=LATENT_DIM, hidden_dim=256).to(DEVICE)
    opt_beta = optim.Adam(model_beta.parameters(), lr=LR)

    # 2. Train for BETA_EPOCHS
    for _ in range(BETA_EPOCHS):
        train_epoch(model_beta, train_loader, opt_beta, beta=beta)

    # 3. Record final test loss
    test_m = eval_loss(model_beta, test_loader, beta=beta)
    beta_results["beta"].append(beta)
    beta_results["recon_loss"].append(test_m["recon_loss"])
    beta_results["kl_loss"].append(test_m["kl_loss"])

    print(
        f"β={beta:.1f}  |  recon_loss={test_m['recon_loss']:.3f}  |  "
        f"kl_loss={test_m['kl_loss']:.3f}"
    )

# ── Plot Pareto frontier ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
for i, beta in enumerate(betas):
    ax.scatter(
        beta_results["kl_loss"][i],
        beta_results["recon_loss"][i],
        s=100,
        label=f"β={beta}",
    )
    ax.annotate(
        f"β={beta}",
        (beta_results["kl_loss"][i], beta_results["recon_loss"][i]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=10,
    )

ax.set_xlabel("KL Divergence → (more regularisation)")
ax.set_ylabel("Reconstruction Loss (BCE) → (better quality ↓)")
ax.set_title("β-VAE: Reconstruction × KL Tradeoff (Pareto Frontier)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

### 6. Latent Space Structure (2D latent only)

When `latent_dim=2`, we can visualise the full latent space directly.
Colour each test point by its digit label and see how the VAE organises
digits in the 2D plane.

A well-trained VAE with 2D latent arranges digits in a **continuous manifold**
where nearby points produce similar digits, and the origin generates
a "central" digit that blends between classes.


In [ ]:
# NOTE: This cell only works when LATENT_DIM = 2.
#       Re-run the notebook from the top with LATENT_DIM = 2 to activate.
#
# Encode the entire test set and plot μ in 2D, coloured by digit label.
# A well-trained VAE organises digits into distinct but connected clusters.

if LATENT_DIM == 2:

    @torch.no_grad()
    def encode_dataset(model, loader):
        """Encode the full dataset and collect μ + labels."""
        model.eval()
        all_mu = []
        all_labels = []
        for x, y in loader:
            x = x.to(DEVICE)
            mu, _ = model.encoder(x)
            all_mu.append(mu.cpu())
            all_labels.append(y)
        return torch.cat(all_mu, dim=0), torch.cat(all_labels, dim=0)

    mu_all, labels_all = encode_dataset(model, test_loader)

    fig, ax = plt.subplots(figsize=(7, 6))
    scatter = ax.scatter(
        mu_all[:, 0],
        mu_all[:, 1],
        c=labels_all,
        cmap="tab10",
        s=8,
        alpha=0.7,
    )
    cbar = fig.colorbar(scatter, ax=ax, ticks=range(10))
    cbar.set_label("Digit label")
    ax.set_xlabel("μ₀")
    ax.set_ylabel("μ₁")
    ax.set_title("VAE latent space (2D) — test set encodings coloured by digit")
    plt.tight_layout()
else:
    print(f"Skipping — LATENT_DIM={LATENT_DIM}, need LATENT_DIM=2")

### Summary

**What you should take away:**

1. **Reparameterisation trick** makes sampling differentiable — the core innovation of VAE
2. **ELBO = Reconstruction - KL** — two competing forces; balance controls what the model learns
3. **Latent space structure** — VAE organises digits smoothly, enabling interpolation and generation
4. **$\beta$ tradeoff** — increasing $\beta$ improves latent space regularity at the cost of blurrier outputs